**Overview**  
This notebook describes folloiwing stages:
1. Data import and reading;
2. Transformation of data: molecules' names are transformed into SMILES, several manual replacements and exclusions are performed;
3. Calculation of standard and PaDEL descriptors;
4. Dropping the most correlated features;
5. Export of descriptors and targets for future use.

In [1]:
from rdkit import RDConfig
from tqdm import tqdm
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from copy import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from sklearn.decomposition import PCA
import pubchempy as pcp
import cirpy
import datamol as dm
import molfeat
from molfeat_padel.calc import PadelDescriptors
from molfeat.calc.descriptors import RDKitDescriptors2D, RDKitDescriptors3D
from molfeat.trans import MoleculeTransformer
import mols2grid
tqdm.pandas()

import pickle
from sklearn.impute import KNNImputer
from feature_engine.selection import DropCorrelatedFeatures 


Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


First, let's read the data. It has a simple format : target (EACN values) vs names of molecules



In [2]:
Data = pd.read_csv('targets.csv', names = ['Name', 'EACN'] )
Data.head()

,Name,EACN
0,1-Bromo-2-methylpropane,-3.3
1,1-Bromo-3-Methylpropane,-3.4
2,1-Chlorodecane,3.5
3,1-Chlorododecane,5.6
4,1-Chlorohexadecane,9.8


Then, let's get SMILES of our molecule with a CIR 

In [3]:
def get_SMILES(name):
    smi = cirpy.resolve(name, 'smiles')
    return smi  

In [4]:
Data['SMILES'] = Data['Name'].progress_apply(get_SMILES)

 48%|██████████████████████████████████████▊                                          | 89/186 [01:35<01:43,  1.07s/it]


KeyboardInterrupt: 

Some names were not transformed into SMILES, let's get them

In [10]:
Data[Data['SMILES'].isnull()]

,Name,EACN,SMILES
54,Cetiol-S,17.00,None
82,Delta-3-Carene,2.45,None
114,Hexamethydisiloxane,12.00,None
130,Methyl CedrylEther,3.50,None


Some names were not found with CIR, thereforem we add SMILES string manually. As Cetiol-S does not have certain formula, we will exlude ot from dataset

In [11]:
Data.loc[82, 'SMILES'] = 'CC1=CCC2C(C1)C2(C)C'
Data.loc[114, 'SMILES'] =   'O([Si](C)(C)C)[Si](C)(C)C'
Data.loc[130, 'SMILES'] =   'CC1CCC2C13CCC(C(C3)C2(C)C)(C)OC'
Data.dropna(axis = 0, inplace = True)

Let's check if there are mixtures in our dataset

In [45]:

indices = Data[Data['SMILES'].str.contains('.', regex = False, na = False)].index
Data.loc[indices]

,Name,EACN,SMILES
66,Cyclohexane Decane,14.5,CCCCCCCCCC.C1CCCCC1
67,Cyclohexane Ethane,3.8,CC.C1CCCCC1
68,Cyclohexane Hexane,17.5,CCCCCC.C1CCCCC1
69,Cyclohexane Octane,7.0,CCCCCCCC.C1CCCCC1
70,Cyclohexane Propane,5.6,CCC.C1CCCCC1
109,Glycerol Tridecanoate,14.0,CCCCCCCCCCCCC(O)=O.OCC(O)CO


There are several instances of alkylcyclohexane and glycerol tridecanoate, which were not tranformed into SMILES properly . We will again replace them manually:

In [47]:

Data.loc[66, 'SMILES'] = 'CCCCCCCCCCC1CCCCC1'
Data.loc[67, 'SMILES'] =  'CCC1CCCCC1'
Data.loc[68, 'SMILES'] = 'CCCCCCC1CCCCC1'
Data.loc[69, 'SMILES'] = 'CCCCCCCCC1CCCCC1'
Data.loc[70, 'SMILES'] ='CCCC1CCCCC1'
Data.loc[109, 'SMILES']= 'CCCCCCCCCC(=O)OCC(OC(=O)CCCCCCCCCC)COC(=O)CCCCCCCCCC'

And save it as *.csv* file

In [50]:
Data.to_csv('targets_with_SMILES_curated.csv')

**Descriptors calculation**

To calculate 3D descriptors, all the molecules should be optimized in 3D space. To do it, we create special function and then apply it to every molecule in Data

In [9]:
Data = pd.read_csv('targets_with_SMILES_curated.csv', index_col=0)[:5]

In [10]:
def embed_optimize(smi):
    try:
        m = Chem.MolFromSmiles(smi)
        m = Chem.AddHs(m)
        params = AllChem.ETKDGv3()
        params.maxIterations = 1000
        params.useRandomCoords = True
        AllChem.EmbedMolecule(m, params)
        AllChem.MMFFOptimizeMolecule(m, maxIters=500)
        return m
    except:
        print(smi)

In [11]:
Data['mol'] = Data.SMILES.progress_apply(embed_optimize)

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 45.20it/s]


Then we create instances of descriptors calculators and molecule transformers and calculate 2D and 3D RDKit descriptors

In [12]:
calc_2d = RDKitDescriptors2D()
calc_3d = RDKitDescriptors3D()
trans_2D = MoleculeTransformer(calc_2d, verbose = True)
trans_3D = MoleculeTransformer(calc_3d, verbose = True)

And then we calculate 2D and 3D descriptors as concatenate them into one DataFrame

In [13]:
with dm.without_rdkit_log():
    feats_2D = trans_2D(Data.mol.values)
    feats_3D = trans_3D(Data.mol.values)

In [14]:
X_standard = pd.DataFrame(np.concatenate([feats_2D, feats_3D], axis  = 1), columns=calc_2d.columns + calc_3d.columns)
X_standard.drop(columns='Alerts', inplace = True)

Let's check for nulls

In [15]:
X_standard.isnull().sum().sum()

0

There are no nulls and NaNs, therefore, we save all RDKit descriptors as is

In [58]:
with open('X_descriptors_initial.pickle', 'wb') as out:
    pickle.dump(X_standard, out)

To create DataSet of **[Padel descriptors](https://onlinelibrary.wiley.com/doi/full/10.1002/jcc.21707)**, we will use add-on for molfeat. 

In [16]:
calc_padel = MoleculeTransformer(featurizer=PadelDescriptors())

In [17]:
df = pd.DataFrame(columns = ['SMILES', 'EACN'] + calc_padel.columns)
for i, (smi, mol, EACN) in tqdm(enumerate(zip(Data.SMILES.values, Data.mol.values, Data['EACN']))):
    df.loc[i, 'SMILES']= smi
    df.loc[i, 'EACN'] = EACN
    df.loc[i, calc_padel.columns] = calc_padel(mol).flatten()


5it [00:10,  2.04s/it]


As some of the data is missing, we can recalculate it with simple scikit-learn [KNN imputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html). We save it for further work

In [18]:
print(df.isnull().sum(axis = 1).sort_values()[::-1].to_string())

4    0
3    0
2    0
1    0
0    0


In [63]:
imputer = KNNImputer()
X_padel = df.iloc[:, 2:].copy()
_cols = X_padel.columns
X_padel = pd.DataFrame(imputer.fit_transform(X_padel), columns = _cols)


In [64]:
with open('imputer.pickle', 'wb') as output:
    pickle.dump(imputer, output)

Then we save PaDEL descriptors in *.csv* format

In [20]:
with open('features_to_drop_Padel.pickle', 'rb') as inp:
    features_to_drop = pickle.load(inp)

In [22]:
df.drop(columns = features_to_drop)

,SMILES,EACN,PaDEL_nAcid,PaDEL_ALogP,PaDEL_ALogp2,PaDEL_AMR,PaDEL_nB,PaDEL_nS,PaDEL_nP,PaDEL_nF,...,PaDEL_PubchemFP871,PaDEL_PubchemFP872,PaDEL_PubchemFP873,PaDEL_PubchemFP874,PaDEL_PubchemFP875,PaDEL_PubchemFP876,PaDEL_PubchemFP877,PaDEL_PubchemFP878,PaDEL_PubchemFP879,PaDEL_PubchemFP880
0,CC(C)CBr,-3.3,0.0,1.4733,2.170613,27.5791,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CCCCBr,-3.4,0.0,0.3757,0.14115,25.0332,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CCCCCCCCCCCl,3.5,0.0,-1.4968,2.24041,39.5193,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CCCCCCCCCCCCCl,5.6,0.0,-2.0728,4.2965,45.3425,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CCCCCCCCCCCCCCCCCl,9.8,0.0,-3.2248,10.399335,56.9889,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
df.loc[0, 'PaDEL_E3s']

0.5243585906681948

In [23]:
with open('X_padel_dropped.pickle', 'rb') as inp:
    X_old = pickle.load(inp)

In [31]:
X_old.loc[0, 'PaDEL_E3s']

0.5243585906681948

In [65]:
with open('X_padel_initial.pickle', 'wb') as output:
    pickle.dump(X_padel, output)

**Getting rid of correlated features**

Both datasets contain a large number of features (852 for standard and 2760 for PaDel), some of them are correlated. It can impart our ML results and will require more computing power and time. We can use DropCorrelated features function of features-engine module with Pearson coefficient threshold value of 0.8 to decrease the number of features and speed up caclulations

In [67]:
print('The number of features in Standard descriptors is {}'.format(X_standard.shape[1]))
print('The number of features in PaDel descriptors is {}'.format(X_padel.shape[1]))

The number of features in Standard descriptors is 852
The number of features in PaDel descriptors is 2756


Here we drop correlated features and save features that we're dropping for Standard dataset

In [68]:
dropper = DropCorrelatedFeatures(threshold=0.8)
X_standard_dropped = dropper.fit_transform(X_standard)
print('The number of features in Standard descriptors is {}'.format(X_standard.shape[1]))
print('The number of features in Standard descriptors after drop  is {}'.format(X_standard_dropped.shape[1]))

The number of features in Standard descriptors is 852
The number of features in Standard descriptors after drop  is 220


In [69]:
with open('features_to_drop_Standard.pickle', 'wb') as output:
    pickle.dump(dropper.features_to_drop_, output)

Here we drop correlated features and save features that we're dropping for PaDEL dataset

In [70]:
dropper = DropCorrelatedFeatures(threshold=0.8)
X_padel_dropped = dropper.fit_transform(X_padel)
print('The number of features in Standard descriptors is {}'.format(X_padel.shape[1]))
print('The number of features in Padel descriptors after drop  is {}'.format(X_padel_dropped.shape[1]))

The number of features in Standard descriptors is 2756
The number of features in Padel descriptors after drop  is 1398


In [71]:
with open('features_to_drop_Padel.pickle', 'wb') as output:
    pickle.dump(dropper.features_to_drop_, output)

We drastically reduced the number of features for RDKit and PaDEL and we save them into pickle files

In [72]:
with open('X_standard_dropped.pickle', 'wb') as output:
    pickle.dump(X_standard_dropped, output)

In [73]:
with open('X_padel_dropped.pickle', 'wb') as output:
    pickle.dump(X_padel_dropped, output)

**Export of target**

We save target values (EACN) for training

In [74]:
EACN = Data['EACN']
with open('EACN.pickle', 'wb') as output:
    pickle.dump(EACN, output)